# 🚀 Attrition Prediction — Model Builder v3
**Key fixes in this version:**
- ✅ Target construction uses `Termination_Date` + `Termination_Reason_Category = 'Voluntary'` as primary source — matches dashboard exactly
- ✅ `HC_LVO_EE` used only as a cross-validation check, never as the label source
- ✅ Active filter (`HC_CTT_EE=1`) applied **after** label assignment, not before
- ✅ Transfer+resign deduplication: keeps reorg-IN row (where `HC_LVO_EE=1` lives)
- ✅ Prior-year org/job-family churn rates (no target leakage)
- ✅ `Days_Since_Last_Promotion` computed correctly per snapshot
- ✅ Threshold optimised on F2-score (recall-weighted) not hardcoded 0.5
- ✅ SHAP TreeExplainer with per-employee top drivers in output

---
## 0. Install Dependencies

In [ ]:
!pip install pandas numpy lightgbm scikit-learn shap joblib matplotlib seaborn pyarrow python-dateutil

---
## 1. Configuration
**Edit the two paths below. Everything else is pre-configured for your schema.**

In [ ]:
import os, json, time, warnings
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import lightgbm as lgb
from dateutil.relativedelta import relativedelta
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score, recall_score, precision_score,
    f1_score, confusion_matrix, precision_recall_curve, average_precision_score
)
from IPython.display import display
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
plt.style.use('seaborn-v0_8-whitegrid')

# ===================== EDIT THESE =====================
PATH_PARQUET  = 'your_headcount_data.parquet'   # ← your parquet file
PATH_OUTPUT   = 'model_artifacts'               # ← output folder
# ======================================================

os.makedirs(PATH_OUTPUT, exist_ok=True)

PREDICTION_HORIZON_MONTHS = 6
TARGET_COLUMN             = 'Will_Leave_Within_6M'
VOLUNTARY_LABEL           = 'Voluntary'   # exact string in Termination_Reason_Category

# Auto-detect test year (most recent year with positive labels).
# Override by setting: TEST_YEAR = 2024
TEST_YEAR = None

# Columns that reveal the future — ALWAYS excluded from features
LEAKAGE_COLUMNS = [
    'HC_Future_Term_Current Year_EE_Vol',
    'HC_Future_Term_Current Year_EE_Invol',
    'HC_Future_Term_Current Year_EE_Other',
    'HC_Future_Term_EE_Current_Year_1_0',
    'HC_Future_Term_EE_Current_Year_3_0',
    'HC_Future_Term_EE',
    'Termination Date', 'Termination Month Number',
    'Termination Reason Category', 'Termination Reason',
    'HC_LVO_EE',          # YTD cumulative — leakage risk if used raw
    'HC_DEC_EE', 'HC_DEC_EE_LTD', 'HC_DEC_EE_INT',
    'HC_DEC_CwkExOffshore', 'HC_DEC_CwkNonworker',
]

MANAGEMENT_LEVEL_MAP = {
    'Admin Business Coordinator':'Admin','Admin Business Lead':'Admin',
    'Admin Business Partner':'Admin','Administrative Assistant':'Admin',
    'Executive Assistant':'Admin','Employee Fixed Term':'Admin',
    'Analyst':'Analyst',
    'Associate':'Associate','Senior Associate':'Associate',
    'Vice President':'VP',
    'Director':'Director','Principal':'Director','Executive Director':'Director',
    'Managing Director':'MD+','Senior Managing Director':'MD+',
    'CEO-Chairman':'MD+','Fund Partner':'MD+','Partner':'MD+',
    'President':'MD+','Senior Advisor':'MD+','Vice Chairman':'MD+',
    'Specialist':'Specialist','Intern':'Intern',
    'Contingent Worker':'Contingent Worker',
}
LEVEL_BENCHMARKS = {
    'Analyst':2.0,'Associate':3.0,'VP':4.0,'Director':4.0,
    'MD+':5.0,'Admin':5.0,'Specialist':3.0,'Intern':1.0,
}
PERFORMANCE_MAP = {
    'Significantly Outperformed':5,'Exceeding':5,
    'Outperformed':4,'Strongly Performed':3,'Tracking':3,
    'Achieved Most Expectations':2,'Needs Improvement':1,
    'New Hire: Needs Improvement':1,
}
print('✅ Configuration loaded.')

---
## 2. Data Loading

In [ ]:
print('='*60)
print('STEP 1: LOADING DATA')
print('='*60)

df_raw = pd.read_parquet(PATH_PARQUET)
df_raw.columns = df_raw.columns.str.strip()
print(f'   Raw rows  : {len(df_raw):,}')
print(f'   Columns   : {df_raw.shape[1]}')

# ── Employee ID cleanup ───────────────────────────────────────────────────
def clean_id(val):
    s = str(val).strip()
    return s[:-2] if s.endswith('.0') else s

id_col = next((c for c in df_raw.columns if 'Employee ID' in c or 'Employee_ID' in c), None)
if id_col:
    df_raw['Employee_ID'] = df_raw[id_col].apply(clean_id)
else:
    raise ValueError("Could not find Employee ID column — check column names.")

# ── Snapshot date ─────────────────────────────────────────────────────────
date_col = 'Snapshot_Date' if 'Snapshot_Date' in df_raw.columns else 'Data As Of'
df_raw['Snapshot_Date'] = pd.to_datetime(df_raw[date_col], errors='coerce')
bad = df_raw['Snapshot_Date'].isna()
if bad.any():
    df_raw.loc[bad, 'Snapshot_Date'] = pd.to_datetime(
        pd.to_numeric(df_raw.loc[bad, date_col], errors='coerce'),
        unit='D', origin='1899-12-30'
    )
df_raw = df_raw.dropna(subset=['Snapshot_Date','Employee_ID'])
df_raw['Snapshot_Date'] = df_raw['Snapshot_Date'].astype('datetime64[ms]')
df_raw['Year']  = df_raw['Snapshot_Date'].dt.year
df_raw['Month'] = df_raw['Snapshot_Date'].dt.month

# ── Normalise binary flags ────────────────────────────────────────────────
FLAG_COLS = [
    'HC_CTT_EE','HC_LVO_EE','HC_HIR_EE','HC_HTT_EE','HC_BSL_EE',
    'HC_RGI_EE','HC_RGO_EE','HC_RTT_EE','HC_XTT_EE','HC_XOT_EE',
    'HC_XIN_EE','HC_LIN_EE','HC_LOT_EE',
    'HC_Did_Not_Start_Hire_EE','HC_Did_Not_Start_Term_EE',
]
for col in FLAG_COLS:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce').fillna(0).abs().clip(0,1).astype(int)

# ── Termination date ──────────────────────────────────────────────────────
term_date_col = next((c for c in df_raw.columns if 'Termination Date' in c or 'Termination_Date' in c), None)
if term_date_col:
    df_raw['Termination_Date'] = pd.to_datetime(df_raw[term_date_col], errors='coerce')
else:
    raise ValueError("'Termination Date' column not found.")

term_reason_col = next((c for c in df_raw.columns if 'Termination Reason Category' in c or 'Termination_Reason_Category' in c), None)
if term_reason_col:
    df_raw['Term_Reason_Cat'] = df_raw[term_reason_col].astype(str).str.strip()
else:
    raise ValueError("'Termination Reason Category' column not found.")

# ── Performance score ─────────────────────────────────────────────────────
if 'Performance_Score' not in df_raw.columns or df_raw['Performance_Score'].isna().mean() > 0.5:
    if 'Performance_Text' in df_raw.columns:
        df_raw['Performance_Score'] = df_raw['Performance_Text'].astype(str).str.strip().map(PERFORMANCE_MAP)

# ── Management level ──────────────────────────────────────────────────────
mgmt_col = next((c for c in df_raw.columns if c.strip() == 'Management Level'), None)
if mgmt_col:
    df_raw['Mgmt_Level_Agg'] = df_raw[mgmt_col].astype(str).str.strip().map(MANAGEMENT_LEVEL_MAP).fillna('Unmapped')
else:
    df_raw['Mgmt_Level_Agg'] = 'Unmapped'

# ── Promoted flag ─────────────────────────────────────────────────────────
if 'Is_Promoted' not in df_raw.columns:
    promo_col = next((c for c in df_raw.columns if 'On Cycle Promotion' in c), None)
    if promo_col:
        df_raw['Is_Promoted'] = df_raw[promo_col].notna().astype(int)
    else:
        df_raw['Is_Promoted'] = 0

print(f'   HC_LVO_EE=1 rows     : {df_raw["HC_LVO_EE"].sum():,}')
print(f'   HC_CTT_EE=1 rows     : {df_raw["HC_CTT_EE"].sum():,}')
print(f'   Voluntary term rows  : {(df_raw["Term_Reason_Cat"]==VOLUNTARY_LABEL).sum():,}')
print(f'   Date range           : {df_raw["Snapshot_Date"].min().date()} → {df_raw["Snapshot_Date"].max().date()}')
print(f'   Years                : {sorted(df_raw["Year"].unique())}')

---
## 3. Target Construction — Leakage-Safe Forward-Looking Label

**Logic (using `Termination_Date` as primary source — matches dashboard):**
1. Identify voluntary leavers: `Term_Reason_Cat == 'Voluntary'` → get their `Termination_Date`
2. For each active employee snapshot: if their `Termination_Date` falls within the next 6 months → label = 1
3. Active filter applied **after** label assignment
4. Most recent 6 months = live scoring population (no future to label)

**Cross-check:** Compare `HC_LVO_EE` count vs `Termination_Date` count to validate gap.

In [ ]:
print('='*60)
print('STEP 2: TARGET CONSTRUCTION')
print('='*60)

df_raw = df_raw.sort_values(['Employee_ID','Snapshot_Date']).reset_index(drop=True)

# ── PRIMARY: Voluntary departure date from Termination_Date ───────────────
# One departure date per employee — their actual last working day
vol_departures = (
    df_raw[df_raw['Term_Reason_Cat'] == VOLUNTARY_LABEL]
    [['Employee_ID','Termination_Date']]
    .dropna(subset=['Termination_Date'])
    .drop_duplicates(subset='Employee_ID', keep='first')  # in case of multiple rows
    .rename(columns={'Termination_Date': 'Vol_Departure_Date'})
)
print(f'   Unique voluntary leavers (Termination_Date): {len(vol_departures):,}')

# ── CROSS-CHECK: compare with HC_LVO_EE count ─────────────────────────────
lvo_count = df_raw[df_raw['HC_LVO_EE']==1]['Employee_ID'].nunique()
term_count = len(vol_departures)
print(f'   HC_LVO_EE unique employees                 : {lvo_count:,}')
print(f'   Gap (in Termination_Date but not LVO_EE)   : {term_count - lvo_count:,}')
if term_count > lvo_count:
    gap_ids = set(vol_departures['Employee_ID']) - set(df_raw[df_raw['HC_LVO_EE']==1]['Employee_ID'])
    print(f'   ↑ These {len(gap_ids):,} employees were missing from HC_LVO_EE — this was your dashboard gap.')

# ── Merge departure date onto ALL rows ────────────────────────────────────
df_raw = pd.merge(df_raw, vol_departures, on='Employee_ID', how='left')

# ── Forward horizon end per snapshot ─────────────────────────────────────
df_raw['Horizon_End'] = df_raw['Snapshot_Date'] + pd.DateOffset(months=PREDICTION_HORIZON_MONTHS)

# ── Label: voluntary departure falls strictly within (Snapshot, Snapshot + 6M] ──
df_raw[TARGET_COLUMN] = (
    (df_raw['Vol_Departure_Date'].notna()) &
    (df_raw['Vol_Departure_Date'] > df_raw['Snapshot_Date']) &
    (df_raw['Vol_Departure_Date'] <= df_raw['Horizon_End'])
).astype(int)

# ── ACTIVE POPULATION FILTER (applied AFTER label assignment) ─────────────
# Keep: active (CTT=1), exclude did-not-starts, exclude contingent workers
dns_hire = df_raw.get('HC_Did_Not_Start_Hire_EE', pd.Series(0, index=df_raw.index))
dns_term = df_raw.get('HC_Did_Not_Start_Term_EE', pd.Series(0, index=df_raw.index))

active_mask = (
    (df_raw['HC_CTT_EE'] == 1) &
    (dns_hire == 0) &
    (dns_term == 0) &
    (df_raw['Mgmt_Level_Agg'] != 'Contingent Worker')
)
df = df_raw[active_mask].copy()

# ── Handle multi-row months (transfer + resign same month) ────────────────
# HC_LVO_EE=1 sits on the reorg-IN row. When deduplicating same employee+month,
# keep the row with highest HC_LVO_EE (i.e. the reorg-in row if they resigned).
# For non-resignation months, all rows have LVO=0 so max() still works correctly.
df['_lvo_sort'] = df.get('HC_LVO_EE', 0)
df = (df
      .sort_values(['Employee_ID','Snapshot_Date','_lvo_sort'], ascending=[True,True,False])
      .drop_duplicates(subset=['Employee_ID','Snapshot_Date'], keep='first')
      .drop(columns=['_lvo_sort'])
)

# ── Live scoring population ───────────────────────────────────────────────
cutoff_date = df['Snapshot_Date'].max() - pd.DateOffset(months=PREDICTION_HORIZON_MONTHS)
df_live = (df[df['Snapshot_Date'] > cutoff_date]
           .sort_values(['Employee_ID','Snapshot_Date'])
           .drop_duplicates(subset='Employee_ID', keep='last')
           .copy())

# ── Model population ──────────────────────────────────────────────────────
df_model = df[df['Snapshot_Date'] <= cutoff_date].copy()

# Drop leakage columns
leak_drop = [c for c in LEAKAGE_COLUMNS if c in df_model.columns]
for ds in [df_model, df_live]:
    ds.drop(columns=leak_drop + ['Vol_Departure_Date','Horizon_End'], inplace=True, errors='ignore')

print(f'\n   Model population  : {len(df_model):,} rows ({df_model["Employee_ID"].nunique():,} unique employees)')
print(f'   Live scoring pop  : {len(df_live):,} unique employees')
print(f'   Target = 1        : {df_model[TARGET_COLUMN].sum():,}')
print(f'   Target rate       : {df_model[TARGET_COLUMN].mean():.2%}')
print(f'   Leakage cols dropped: {len(leak_drop)}')

---
## 4. Feature Engineering

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values(['Employee_ID','Snapshot_Date'])

    # ── 1. TENURE ────────────────────────────────────────────────────────────
    if 'Length of Service' in df.columns:
        df['Total_Tenure_Years'] = pd.to_numeric(df['Length of Service'], errors='coerce')
    else:
        cs_col = next((c for c in df.columns if 'Continuous Service Date' in c), None)
        if cs_col:
            df[cs_col] = pd.to_datetime(df[cs_col], errors='coerce')
            df['Total_Tenure_Years'] = (df['Snapshot_Date'] - df[cs_col]).dt.days / 365.25
        else:
            df['Total_Tenure_Years'] = 0
    df['Total_Tenure_Years'] = df['Total_Tenure_Years'].clip(0).fillna(df['Total_Tenure_Years'].median())
    df['Tenure_Band'] = pd.cut(
        df['Total_Tenure_Years'],
        bins=[0,1,2,4,6,10,999],
        labels=['<1yr','1-2yr','2-4yr','4-6yr','6-10yr','10+yr'], right=False
    ).astype(str)

    tit_col = next((c for c in df.columns if 'Time In Title' in c), None)
    df['Time_In_Title_Current'] = pd.to_numeric(df[tit_col], errors='coerce').fillna(0) if tit_col else 0

    # Years in level + stuck signal
    df['_first_in_level'] = df.groupby(['Employee_ID','Mgmt_Level_Agg'])['Snapshot_Date'].transform('min')
    df['_years_in_level']  = (df['Snapshot_Date'] - df['_first_in_level']).dt.days / 365.25
    df['_expected_years']  = df['Mgmt_Level_Agg'].map(LEVEL_BENCHMARKS).fillna(3.0)
    df['Cycles_In_Level']  = (df['_years_in_level'] / df['_expected_years']).clip(0)
    df['Is_Stuck_In_Level']= (df['Cycles_In_Level'] >= 1.5).astype(int)
    df.drop(columns=['_first_in_level','_years_in_level','_expected_years'], inplace=True, errors='ignore')

    # Days since last promotion — correct per-snapshot computation
    if 'Is_Promoted' in df.columns:
        promo_events = df[df['Is_Promoted']==1][['Employee_ID','Snapshot_Date']].rename(columns={'Snapshot_Date':'Promo_Date'})
        merged = pd.merge(df[['Employee_ID','Snapshot_Date']], promo_events, on='Employee_ID', how='left')
        merged = merged[merged['Promo_Date'] < merged['Snapshot_Date']]
        last_promo = merged.groupby(['Employee_ID','Snapshot_Date'])['Promo_Date'].max().reset_index()
        last_promo['Days_Since_Last_Promotion'] = (last_promo['Snapshot_Date'] - last_promo['Promo_Date']).dt.days
        df = pd.merge(df, last_promo[['Employee_ID','Snapshot_Date','Days_Since_Last_Promotion']],
                      on=['Employee_ID','Snapshot_Date'], how='left')
        df['Days_Since_Last_Promotion'] = df['Days_Since_Last_Promotion'].fillna(-1)
    else:
        df['Is_Promoted'] = 0
        df['Days_Since_Last_Promotion'] = -1

    # ── 2. PERFORMANCE ────────────────────────────────────────────────────────
    if 'Performance_Score' not in df.columns:
        df['Performance_Score'] = np.nan
    df['Is_High_Performer'] = df['Performance_Score'].isin([4,5]).astype(int)
    df['Is_Low_Performer']  = (df['Performance_Score'] <= 2).astype(int)

    # YoY drop
    perf_ref = df.drop_duplicates(subset=['Employee_ID','Year'])[['Employee_ID','Year','Performance_Score']].copy()
    perf_ref = perf_ref.rename(columns={'Year':'_py','Performance_Score':'_prev_perf'})
    perf_ref['Year'] = perf_ref['_py'] + 1
    df = pd.merge(df, perf_ref[['Employee_ID','Year','_prev_perf']], on=['Employee_ID','Year'], how='left')
    df['Performance_Drop_Flag'] = ((df['_prev_perf'].notna()) & (df['Performance_Score'] < df['_prev_perf'])).astype(int)
    df.drop(columns=['_prev_perf'], inplace=True, errors='ignore')

    # Rolling 3yr avg + trajectory slope
    yp = (df.drop_duplicates(subset=['Employee_ID','Year'])[['Employee_ID','Year','Performance_Score']]
            .copy().sort_values(['Employee_ID','Year']))
    yp['Perf_Rolling_3Yr'] = yp.groupby('Employee_ID')['Performance_Score'].transform(
        lambda x: x.rolling(3, min_periods=1).mean())

    def slope(s):
        v = s.dropna()
        if len(v) < 2: return 0.0
        x = np.arange(len(v), dtype=float)
        xm,ym = x.mean(), v.values.astype(float).mean()
        d = ((x-xm)**2).sum()
        return float(((x-xm)*(v.values.astype(float)-ym)).sum()/d) if d!=0 else 0.0

    yp['Perf_Trajectory_Slope'] = yp.groupby('Employee_ID')['Performance_Score'].transform(
        lambda x: x.expanding(min_periods=2).apply(slope, raw=False)).fillna(0)

    # Consecutive low perf years
    yp['_is_low'] = (yp['Performance_Score'] <= 2).astype(int)
    yp['Consec_Low_Perf'] = yp.groupby('Employee_ID')['_is_low'].transform(
        lambda x: x * (x.groupby((x != x.shift()).cumsum()).cumcount() + 1))

    df = pd.merge(df, yp[['Employee_ID','Year','Perf_Rolling_3Yr','Perf_Trajectory_Slope','Consec_Low_Perf']],
                  on=['Employee_ID','Year'], how='left')
    df[['Perf_Rolling_3Yr','Perf_Trajectory_Slope','Consec_Low_Perf']] = (
        df[['Perf_Rolling_3Yr','Perf_Trajectory_Slope','Consec_Low_Perf']].fillna(0))

    # ── 3. PEER DYNAMICS ─────────────────────────────────────────────────────
    org_col = next((c for c in df.columns if 'Level 4 Org' in c), None) or '_org4'
    if '_org4' not in df.columns and org_col == '_org4':
        df['_org4'] = 'Unknown'
    promo_cnt = df.groupby(['Year','Mgmt_Level_Agg',org_col])['Is_Promoted'].transform('sum')
    hc_cnt    = df.groupby(['Year','Mgmt_Level_Agg',org_col])['Employee_ID'].transform('count')
    df['Peer_Promo_Rate']    = (promo_cnt / hc_cnt.replace(0,1)).fillna(0)
    df['Peer_Pressure_Flag'] = ((df['Is_Promoted']==0) & (df['Peer_Promo_Rate']>0.10)).astype(int)

    # ── 4. MANAGER SIGNALS ────────────────────────────────────────────────────
    mgr_id_col = next((c for c in df.columns if 'Manager ID' in c or 'Manager_ID' in c), None)
    span_col   = next((c for c in df.columns if 'Active Manager Direct EMP' in c and 'CWK' not in c and 'All' not in c), None)
    if mgr_id_col:
        df[mgr_id_col] = df[mgr_id_col].astype(str).str.strip().str.replace(r'\.0$','',regex=True)
        df['Manager_Span_Of_Control'] = (pd.to_numeric(df[span_col], errors='coerce').fillna(0) if span_col
                                         else df.groupby([mgr_id_col,'Snapshot_Date'])['Employee_ID'].transform('count'))
        mgr_perf = df[['Employee_ID','Snapshot_Date','Performance_Score']].copy()
        mgr_perf.columns = [mgr_id_col,'Snapshot_Date','Mgr_Perf_Score']
        mgr_perf = mgr_perf.drop_duplicates(subset=[mgr_id_col,'Snapshot_Date'])
        df = pd.merge(df, mgr_perf, on=[mgr_id_col,'Snapshot_Date'], how='left')
        df['Mgr_Perf_Score']    = df['Mgr_Perf_Score'].fillna(3)
        df['Performer_vs_Mgr']  = (df['Performance_Score'] / df['Mgr_Perf_Score'].replace(0,1)).fillna(1)
        df['_prev_mgr'] = df.groupby('Employee_ID')[mgr_id_col].shift(1)
        df['_mgr_chg']  = ((df['_prev_mgr'].notna()) & (df[mgr_id_col] != df['_prev_mgr'])).astype(int)
        df['Mgr_Changed_6M'] = df.groupby('Employee_ID')['_mgr_chg'].transform(
            lambda x: x.rolling(6,min_periods=1).max()).fillna(0).astype(int)
        df.drop(columns=['_prev_mgr','_mgr_chg'], inplace=True, errors='ignore')
    else:
        for c in ['Manager_Span_Of_Control','Mgr_Perf_Score','Performer_vs_Mgr','Mgr_Changed_6M']:
            df[c] = 0

    # ── 5. ORG CHURN RATES (prior-year only — no leakage) ────────────────────
    if TARGET_COLUMN in df.columns:
        jf_col = next((c for c in df.columns if c.strip() == 'Job Family'), None)
        if jf_col:
            jf_rates = (df.groupby(['Year',jf_col])[TARGET_COLUMN].mean()
                          .reset_index().rename(columns={TARGET_COLUMN:'JF_Churn_Rate','Year':'_py'}))
            jf_rates['Year'] = jf_rates['_py'] + 1
            df = pd.merge(df, jf_rates[['Year',jf_col,'JF_Churn_Rate']], on=['Year',jf_col], how='left')
            df['JF_Churn_Rate'] = df['JF_Churn_Rate'].fillna(0)
        else:
            df['JF_Churn_Rate'] = 0

        if org_col in df.columns:
            org_rates = (df.groupby(['Year',org_col])[TARGET_COLUMN].mean()
                           .reset_index().rename(columns={TARGET_COLUMN:'Org_Churn_Rate','Year':'_py'}))
            org_rates['Year'] = org_rates['_py'] + 1
            df = pd.merge(df, org_rates[['Year',org_col,'Org_Churn_Rate']], on=['Year',org_col], how='left')
            df['Org_Churn_Rate'] = df['Org_Churn_Rate'].fillna(0)
        else:
            df['Org_Churn_Rate'] = 0
    else:
        df['JF_Churn_Rate'] = df['Org_Churn_Rate'] = 0

    # ── 6. CONTEXT & TEMPORAL ─────────────────────────────────────────────────
    df['Is_Lateral_Hire']         = pd.to_numeric(df.get('HC_Hire_Lateral_Comm',0), errors='coerce').fillna(0).clip(0,1).astype(int)
    df['Is_Campus_Hire']          = pd.to_numeric(df.get('HC_Hire_Campus_Comm',0),  errors='coerce').fillna(0).clip(0,1).astype(int)
    intl_col = next((c for c in df.columns if 'International Assignment Flag' in c), None)
    df['Has_Intl_Assignment']     = df[intl_col].astype(str).str.lower().isin(['yes','y','1','true']).astype(int) if intl_col else 0
    acq_col  = next((c for c in df.columns if 'Acquisition Company' in c), None)
    df['Is_Acquisition_Employee'] = (df[acq_col].notna() & df[acq_col].astype(str).str.strip().ne('') & df[acq_col].astype(str).str.lower().ne('nan')).astype(int) if acq_col else 0
    df['FTE_Value']               = pd.to_numeric(df.get('FTE',1.0), errors='coerce').fillna(1.0).clip(0,1)
    df['Is_People_Manager']       = (df.get('Manager_Span_Of_Control',0) > 0).astype(int)

    for flag in ['HC_RGI_EE','HC_RGO_EE','HC_XIN_EE','HC_XOT_EE']:
        if flag not in df.columns: df[flag] = 0
    df['_rg'] = ((df['HC_RGI_EE']==1)|(df['HC_RGO_EE']==1)).astype(int)
    df['_xf'] = ((df['HC_XIN_EE']==1)|(df['HC_XOT_EE']==1)).astype(int)
    df['Is_Reorg_Recent_6M']    = df.groupby('Employee_ID')['_rg'].transform(lambda x: x.rolling(6,min_periods=1).max()).fillna(0).astype(int)
    df['Is_Transfer_Recent_6M'] = df.groupby('Employee_ID')['_xf'].transform(lambda x: x.rolling(6,min_periods=1).max()).fillna(0).astype(int)
    df.drop(columns=['_rg','_xf','_org4'], inplace=True, errors='ignore')

    df['Month_Sin']            = np.sin(2*np.pi*df['Month']/12)
    df['Month_Cos']            = np.cos(2*np.pi*df['Month']/12)
    df['Is_Post_Bonus_Window'] = df['Month'].isin([2,3,4]).astype(int)
    df['Is_Year_End_Window']   = df['Month'].isin([10,11,12]).astype(int)

    return df

print('⚙️  Engineering features...')
t0 = time.time()
df_model = engineer_features(df_model)
df_live  = engineer_features(df_live)
print(f'✅ Done in {time.time()-t0:.1f}s. Shape: {df_model.shape}')

---
## 5. Feature Selection & Train/Test Split

In [ ]:
NUMERIC_FEATURES = [
    'Total_Tenure_Years','Time_In_Title_Current','Cycles_In_Level',
    'Is_Stuck_In_Level','Days_Since_Last_Promotion','Consec_Low_Perf',
    'Performance_Score','Is_High_Performer','Is_Low_Performer',
    'Performance_Drop_Flag','Perf_Rolling_3Yr','Perf_Trajectory_Slope',
    'Peer_Promo_Rate','Peer_Pressure_Flag','Is_Promoted',
    'Mgr_Perf_Score','Performer_vs_Mgr','Mgr_Changed_6M','Manager_Span_Of_Control',
    'JF_Churn_Rate','Org_Churn_Rate',
    'Is_Lateral_Hire','Is_Campus_Hire','Has_Intl_Assignment',
    'Is_Acquisition_Employee','FTE_Value','Is_People_Manager',
    'Is_Reorg_Recent_6M','Is_Transfer_Recent_6M',
    'Month_Sin','Month_Cos','Is_Post_Bonus_Window','Is_Year_End_Window',
]
CATEGORICAL_FEATURES = ['Mgmt_Level_Agg','Tenure_Band']
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

available_num = [f for f in NUMERIC_FEATURES if f in df_model.columns]
available_cat = [f for f in CATEGORICAL_FEATURES if f in df_model.columns]
available_all = available_num + available_cat
print(f'Features available: {len(available_all)} / {len(ALL_FEATURES)}')

# Time-based split
if TEST_YEAR is None:
    valid_yrs = sorted(df_model[df_model[TARGET_COLUMN]>0]['Year'].unique())
    TEST_YEAR = int(valid_yrs[-1]) if valid_yrs else int(df_model['Year'].max())
print(f'Test year : {TEST_YEAR}')
print(f'Train years: {sorted(y for y in df_model["Year"].unique() if y < TEST_YEAR)}')

# Fill nulls
for ds in [df_model, df_live]:
    for c in available_num:
        if c in ds.columns:
            ds[c] = ds[c].fillna(df_model[c].median() if c in df_model.columns else 0)
    for c in available_cat:
        if c in ds.columns:
            ds[c] = ds[c].fillna('Unknown').astype('category')

df_train = df_model[df_model['Year'] < TEST_YEAR].copy()
df_test  = (df_model[df_model['Year'] == TEST_YEAR]
            .sort_values(['Employee_ID','Snapshot_Date'],ascending=[True,False])
            .drop_duplicates(subset='Employee_ID',keep='first').copy())

X_train, y_train = df_train[available_all], df_train[TARGET_COLUMN]
X_test,  y_test  = df_test[available_all],  df_test[TARGET_COLUMN]
X_live           = df_live[available_all]
groups_train     = df_train['Employee_ID']

print(f'Train: {len(X_train):,} rows | +ve: {y_train.sum():,} ({y_train.mean():.2%})')
print(f'Test : {len(X_test):,} rows  | +ve: {y_test.sum():,} ({y_test.mean():.2%})')
print(f'Live : {len(X_live):,} employees to score')

---
## 6. Train LightGBM

In [ ]:
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos = n_neg / max(n_pos, 1)
print(f'Class ratio (neg/pos): {scale_pos:.1f}x')

lgbm_params = {
    'objective':'binary','metric':'auc',
    'n_estimators':500,'learning_rate':0.05,
    'num_leaves':63,'min_child_samples':50,
    'subsample':0.8,'colsample_bytree':0.8,
    'reg_alpha':0.1,'reg_lambda':0.1,
    'scale_pos_weight':scale_pos,
    'random_state':42,'n_jobs':-1,'verbose':-1,
}

print('\n📊 GroupKFold CV (5 folds) — no employee leakage across folds...')
gkf = GroupKFold(n_splits=5)
cv_results = []
for fold,(tr_idx,val_idx) in enumerate(gkf.split(X_train,y_train,groups=groups_train),1):
    X_tr,X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr,y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    if y_val.sum()==0 or y_tr.sum()==0: continue
    spw = (len(y_tr)-y_tr.sum())/max(y_tr.sum(),1)
    m = lgb.LGBMClassifier(**{**lgbm_params,'scale_pos_weight':spw})
    m.fit(X_tr,y_tr,categorical_feature=available_cat)
    yp = m.predict_proba(X_val)[:,1]
    auc = roc_auc_score(y_val,yp)
    ap  = average_precision_score(y_val,yp)
    cv_results.append({'fold':fold,'auc':auc,'ap':ap})
    print(f'   Fold {fold}: AUC={auc:.4f}  AP={ap:.4f}')

cv_df = pd.DataFrame(cv_results)
print(f'\n   Mean AUC : {cv_df["auc"].mean():.3f} ± {cv_df["auc"].std():.3f}')
print(f'   Mean AP  : {cv_df["ap"].mean():.3f} ± {cv_df["ap"].std():.3f}')

print('\n🚀 Training final model...')
t0 = time.time()
model = lgb.LGBMClassifier(**lgbm_params)
model.fit(X_train,y_train,categorical_feature=available_cat)
print(f'✅ Trained in {time.time()-t0:.1f}s')

---
## 7. Evaluation — Threshold Optimisation + Metrics

In [ ]:
y_prob = model.predict_proba(X_test)[:,1]

# Optimise threshold using F2 (recall weighted 2x over precision)
precs,recs,thresholds = precision_recall_curve(y_test,y_prob)
beta = 2
f_beta = (1+beta**2)*(precs*recs)/(beta**2*precs+recs+1e-9)
best_idx = np.argmax(f_beta)
THRESHOLD = float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.5
y_pred = (y_prob > THRESHOLD).astype(int)

test_auc = roc_auc_score(y_test,y_prob)
test_ap  = average_precision_score(y_test,y_prob)
test_rec = recall_score(y_test,y_pred,zero_division=0)
test_pre = precision_score(y_test,y_pred,zero_division=0)

print('='*50)
print(f'TEST SET RESULTS ({TEST_YEAR})')
print('='*50)
print(f'Optimised threshold : {THRESHOLD:.3f}')
print(f'AUC-ROC             : {test_auc:.4f}')
print(f'AP (PR-AUC)         : {test_ap:.4f}')
print(f'Recall              : {test_rec:.4f}  ← % of real leavers caught')
print(f'Precision           : {test_pre:.4f}  ← % of flagged who actually left')
print(f'Population: {len(X_test):,} | Leavers: {int(y_test.sum()):,} | Flagged: {int(y_pred.sum()):,}')

print('\nRecall @ Risk Tiers:')
for pct in [5,10,15,20]:
    k = max(1,int(len(y_prob)*pct/100))
    top_k = np.argsort(y_prob)[::-1][:k]
    rk = y_test.iloc[top_k].sum()/max(y_test.sum(),1)
    print(f'  Top {pct:2d}%: {rk:.2%} ({y_test.iloc[top_k].sum():.0f}/{y_test.sum():.0f} leavers in top {k:,} employees)')

fig,axes = plt.subplots(1,3,figsize=(18,5))
cm = confusion_matrix(y_test,y_pred)
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=axes[0],
            xticklabels=['Stay','Leave'],yticklabels=['Stay','Leave'])
axes[0].set_title(f'Confusion Matrix ({TEST_YEAR})')
axes[1].hist(y_prob[y_test==0],bins=60,alpha=0.6,label='Stayers',color='steelblue',density=True)
axes[1].hist(y_prob[y_test==1],bins=60,alpha=0.6,label='Leavers',color='salmon',density=True)
axes[1].axvline(THRESHOLD,color='red',linestyle='--',label=f'Threshold={THRESHOLD:.2f}')
axes[1].set_title('Risk Score Distribution'); axes[1].legend()
axes[2].plot(recs,precs,color='darkorange',lw=2,label=f'AP={test_ap:.3f}')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve'); axes[2].legend()
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT,'evaluation_plots.png'),dpi=150,bbox_inches='tight')
plt.show()

---
## 8. SHAP Explainability

In [ ]:
explainer = shap.TreeExplainer(model)
sample_size = min(2000, len(X_test))
X_shap = X_test.sample(sample_size,random_state=42)
print(f'Computing SHAP on {sample_size:,} samples...')
t0 = time.time()
shap_values = explainer.shap_values(X_shap)
shap_pos = shap_values[1] if isinstance(shap_values,list) else shap_values
print(f'Done in {time.time()-t0:.1f}s')

plt.figure(figsize=(10,8))
shap.summary_plot(shap_pos,X_shap,plot_type='bar',max_display=20,show=False)
plt.title('Top 20 Attrition Risk Drivers (SHAP Mean |Value|)')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT,'shap_importance.png'),dpi=150,bbox_inches='tight')
plt.show()

plt.figure(figsize=(10,8))
shap.summary_plot(shap_pos,X_shap,max_display=20,show=False)
plt.title('SHAP Beeswarm — Direction & Magnitude')
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT,'shap_beeswarm.png'),dpi=150,bbox_inches='tight')
plt.show()

mean_shap = pd.DataFrame({'Feature':X_shap.columns,'Mean_SHAP':np.abs(shap_pos).mean(axis=0)}).sort_values('Mean_SHAP',ascending=False)
print('\nTop 10 risk drivers:')
display(mean_shap.head(10))

---
## 9. Live Scoring & Power BI Export

In [ ]:
live_probs = model.predict_proba(X_live)[:,1]
p10 = np.percentile(live_probs,90)
p25 = np.percentile(live_probs,75)
risk_tier = lambda p: 'High' if p>=p10 else ('Medium' if p>=p25 else 'Low')

print('Computing per-employee SHAP drivers...')
live_shap_vals = explainer.shap_values(X_live)
live_shap = live_shap_vals[1] if isinstance(live_shap_vals,list) else live_shap_vals
shap_df = pd.DataFrame(live_shap,columns=X_live.columns)

def top3(row):
    top = row.abs().nlargest(3).index.tolist()
    return pd.Series({f'Top_Driver_{i+1}':f'{f} ({row[f]:+.3f})' for i,f in enumerate(top)})

driver_cols = shap_df.apply(top3,axis=1)

out_cols = ['Employee_ID']
for c in ['Candidate Name','Management Level','Mgmt_Level_Agg','Job Family',
          'Location','Region','Country','Level 2 Org','Level 3 Org','Level 4 Org',
          'Performance_Score','Total_Tenure_Years','Year','Snapshot_Date']:
    if c in df_live.columns: out_cols.append(c)

df_scored = df_live[out_cols].reset_index(drop=True).copy()
df_scored['Risk_Score'] = (live_probs*100).round(1)
df_scored['Risk_Tier']  = [risk_tier(p) for p in live_probs]
df_scored = pd.concat([df_scored, driver_cols.reset_index(drop=True)],axis=1)
df_scored = df_scored.sort_values('Risk_Score',ascending=False)

df_scored.to_csv(os.path.join(PATH_OUTPUT,'attrition_risk_scores.csv'),index=False)
df_scored.to_parquet(os.path.join(PATH_OUTPUT,'attrition_risk_scores.parquet'),index=False)

tc = df_scored['Risk_Tier'].value_counts()
print(f'\nScored {len(df_scored):,} active employees')
print(f'High   (top 10%): {tc.get("High",0):,}  score ≥ {p10*100:.0f}')
print(f'Medium (top 25%): {tc.get("Medium",0):,}  score ≥ {p25*100:.0f}')
print(f'Low             : {tc.get("Low",0):,}')
print(f'\n✅ Scores saved to {PATH_OUTPUT}/')
display(df_scored.head(10))

---
## 10. Save Model Artifacts

In [ ]:
joblib.dump(model,os.path.join(PATH_OUTPUT,'lgbm_attrition_model.joblib'))
with open(os.path.join(PATH_OUTPUT,'feature_names.json'),'w') as f:
    json.dump(available_all,f,indent=2)
with open(os.path.join(PATH_OUTPUT,'model_metadata.json'),'w') as f:
    json.dump({
        'model_type':'LightGBM',
        'prediction_horizon_months':PREDICTION_HORIZON_MONTHS,
        'target_label_source':'Termination_Date + Termination_Reason_Category=Voluntary',
        'test_year':TEST_YEAR,
        'optimised_threshold':round(THRESHOLD,4),
        'test_auc':round(test_auc,4),
        'test_recall':round(test_rec,4),
        'test_precision':round(test_pre,4),
        'cv_mean_auc':round(float(cv_df["auc"].mean()),4),
        'n_features':len(available_all),
        'train_rows':len(X_train),
        'train_positive_rate':round(float(y_train.mean()),4),
    },f,indent=2)
mean_shap.to_csv(os.path.join(PATH_OUTPUT,'shap_feature_importance.csv'),index=False)
print('\n🎉 All artifacts saved:')
for f in sorted(os.listdir(PATH_OUTPUT)):
    sz = os.path.getsize(os.path.join(PATH_OUTPUT,f))
    print(f'  {f:45s} {sz/1024:>8.1f} KB')